# Phase 2: Data Ingestion - All Three Sources


## What This Phase Does

Phase 2 builds the data foundation for the entire project. I am collecting three distinct types of raw data:

| Source | What l will collect | How |
|--------|-----------------|-----|
| VIX (already from Phase 1) | Daily fear index close prices | `yfinance` |
| Macro indicators | 6 Federal Reserve economic series | `fredapi` |
| News headlines | Financial headlines with timestamps | `newsapi` + Kaggle dataset |


In [ ]:
import os
import time
import warnings
import pandas as pd
import numpy as np
import yfinance as yf
from fredapi import Fred
from datetime import datetime, timedelta
from dotenv import load_dotenv

warnings.filterwarnings('ignore')


load_dotenv('../.env')

FRED_API_KEY   = os.getenv('FRED_API_KEY')
NEWS_API_KEY    = os.getenv('NEWS_API_KEY')


print('=== API Key Status ===')
print(f"FRED_API_KEY:  {'LOADED' if FRED_API_KEY  else 'MISSING — check your .env file'}")
print(f"NEWS_API_KEY:   {'LOADED' if NEWS_API_KEY   else 'MISSING — check your .env file (optional if using Kaggle)'}")


os.makedirs('../data/raw',       exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../figures',        exist_ok=True)
os.makedirs('../models',         exist_ok=True)
os.makedirs('../logs',           exist_ok=True)

print('\nDirectories ready.')
print(f'Session started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

=== API Key Status ===
FRED_API_KEY:  LOADED
NEWS_API_KEY:   LOADED

Directories ready.
Session started: 2026-03-27 14:37:28


In [5]:
VIX_PATH = '../data/raw/vix_raw.csv'

if os.path.exists(VIX_PATH):
    # Loading existing Phase 1 file
    vix_df = pd.read_csv(VIX_PATH, index_col=0, parse_dates=True)
    last_date = vix_df.index[-1]
    today     = pd.Timestamp.today().normalize()
    print(f'Loaded existing VIX data: {len(vix_df)} rows')
    print(f'Last date in file: {last_date.date()}')

    # Checking if update needed (more than 1 trading day behind)
    if (today - last_date).days > 1:
        print(f'Updating VIX from {last_date.date()} to {today.date()}...')
        new_data = yf.download('^VIX', start=last_date + timedelta(days=1), progress=False)
        if len(new_data) > 0:
            new_data.columns = ['_'.join(col).strip() for col in new_data.columns.values]
            new_data = new_data[['Close_^VIX']].rename(columns={'Close_^VIX': 'VIX'})
            new_data = new_data.dropna()
            vix_df = pd.concat([vix_df, new_data])
            vix_df = vix_df[~vix_df.index.duplicated(keep='last')]
            vix_df.to_csv(VIX_PATH)
            print(f'Added {len(new_data)} new rows. Total: {len(vix_df)}')
        else:
            print('No new data available yet.')
    else:
        print('VIX data is up to date.')
else:
    # Full download if Phase 1 file missing
    print('Phase 1 file not found — downloading full VIX history...')
    raw = yf.download('^VIX', start='2000-01-01', progress=False)
    raw.columns = ['_'.join(col).strip() for col in raw.columns.values]
    vix_df = raw[['Close_^VIX']].rename(columns={'Close_^VIX': 'VIX'}).dropna()
    vix_df.index.name = 'Date'
    vix_df.to_csv(VIX_PATH)
    print(f'Downloaded and saved: {len(vix_df)} rows')

print(f'\nVIX range: {vix_df.index[0].date()} to {vix_df.index[-1].date()}')
print(f'Data freshness: {(pd.Timestamp.today() - vix_df.index[-1]).days} calendar day(s) behind today')

Loaded existing VIX data: 6598 rows
Last date in file: 2026-03-27
VIX data is up to date.

VIX range: 2000-01-03 to 2026-03-27
Data freshness: 0 calendar day(s) behind today


In [7]:

fred = Fred(api_key=FRED_API_KEY)

# --- helper: retry wrapper -----------------------------------------
def fetch_fred_series(fred_client, series_id, start_date='2000-01-01', max_retries=3):
    """
    Fetch a FRED series with automatic retry on failure.
    Returns a pandas Series, or None if all retries fail.
    """
    for attempt in range(1, max_retries + 1):
        try:
            data = fred_client.get_series(series_id, observation_start=start_date)
            return data
        except Exception as e:
            print(f'  Attempt {attempt}/{max_retries} failed for {series_id}: {e}')
            if attempt < max_retries:
                time.sleep(2 ** attempt)  # exponential backoff: 2s, 4s, 8s
    print(f'  ERROR: Could not fetch {series_id} after {max_retries} attempts.')
    return None

# fetching all 6 series
FRED_SERIES = {
    'fedfunds': 'FEDFUNDS',
    'cpi':      'CPIAUCSL',
    'unrate':   'UNRATE',
    'gs10':     'GS10',
    'indpro':   'INDPRO',
    'm2sl':     'M2SL',
}

macro_frames = {}
print('=== Fetching FRED Macro Data ===')
for col_name, series_id in FRED_SERIES.items():
    s = fetch_fred_series(fred, series_id)
    if s is not None:
        macro_frames[col_name] = s
        last = s.dropna().index[-1]
        print(f'  {series_id:10s} → {len(s):5d} obs, latest: {last.date()}')

# 
macro_df = pd.DataFrame(macro_frames)
macro_df.index.name = 'date'
macro_df.index = pd.to_datetime(macro_df.index)


macro_df.to_csv('../data/raw/macro_raw.csv')
print(f'\nSaved: data/raw/macro_raw.csv')
print(f'Shape: {macro_df.shape} (rows × columns)')
print(f'Date range: {macro_df.index[0].date()} to {macro_df.index[-1].date()}')
macro_df.tail()

=== Fetching FRED Macro Data ===
  FEDFUNDS   →   314 obs, latest: 2026-02-01
  CPIAUCSL   →   314 obs, latest: 2026-02-01
  UNRATE     →   314 obs, latest: 2026-02-01
  GS10       →   314 obs, latest: 2026-02-01
  INDPRO     →   314 obs, latest: 2026-02-01
  M2SL       →   314 obs, latest: 2026-02-01

Saved: data/raw/macro_raw.csv
Shape: (314, 6) (rows × columns)
Date range: 2000-01-01 to 2026-02-01


,fedfunds,cpi,unrate,gs10,indpro,m2sl
date,,,,,,
2025-10-01,4.09,NaN,NaN,4.06,101.2075,22250.4
2025-11-01,3.88,325.063,4.5,4.09,101.3605,22296.5
2025-12-01,3.72,326.031,4.4,4.14,101.6781,22386.9
2026-01-01,3.64,326.588,4.3,4.21,102.3963,22469.1
2026-02-01,3.64,327.460,4.4,4.13,102.5510,22667.3


In [9]:
#FRED Data Quality Check
print('=== FRED Data Quality Report ===')
print(f'{"Series":<12} {"Rows":>6} {"NaN count":>10} {"First date":>12} {"Last date":>12} {"Status":>10}')
print('-' * 70)

all_ok = True
for col in macro_df.columns:
    n_rows  = len(macro_df[col])
    n_nan   = macro_df[col].isna().sum()
    first   = macro_df[col].dropna().index[0].date()
    last    = macro_df[col].dropna().index[-1].date()
    # Flag if more than 3 months of recent data is missing
    days_behind = (pd.Timestamp.today() - macro_df[col].dropna().index[-1]).days
    status  = 'OK' if days_behind < 90 else 'STALE'
    if status != 'OK':
        all_ok = False
    print(f'{col:<12} {n_rows:>6} {n_nan:>10} {str(first):>12} {str(last):>12} {status:>10}')

print('-' * 70)
if all_ok:
    print('All FRED series are current and complete.')
else:
    print('WARNING: Some series may be stale. FRED often has a 1-2 month reporting lag — this is normal.')

=== FRED Data Quality Report ===
Series         Rows  NaN count   First date    Last date     Status
----------------------------------------------------------------------
fedfunds        314          0   2000-01-01   2026-02-01         OK
cpi             314          1   2000-01-01   2026-02-01         OK
unrate          314          1   2000-01-01   2026-02-01         OK
gs10            314          0   2000-01-01   2026-02-01         OK
indpro          314          0   2000-01-01   2026-02-01         OK
m2sl            314          0   2000-01-01   2026-02-01         OK
----------------------------------------------------------------------
All FRED series are current and complete.


In [12]:
from newsapi import NewsApiClient

#NewsAPI: Recent Headlines (Last 30 Days)
newsapi_headlines = []

if not NEWS_API_KEY:
    print('NEWSAPI_KEY not found — skipping recent headlines fetch.')
    print('Will rely on Kaggle historical dataset in Cell 7.')
else:
    newsapi = NewsApiClient(api_key=NEWS_API_KEY)

    # Searching terms related to market stress and VIX drivers
    SEARCH_QUERIES = [
        'VIX volatility index',
        'Federal Reserve interest rates',
        'stock market crash recession',
        'inflation CPI unemployment',
        'S&P 500 market selloff',
    ]

    print('=== Fetching NewsAPI Recent Headlines ===')
    for query in SEARCH_QUERIES:
        for attempt in range(1, 4):  # retry up to 3 times
            try:
                response = newsapi.get_everything(
                    q=query,
                    language='en',
                    sort_by='publishedAt',
                    page_size=100,
                    from_param=(datetime.now() - timedelta(days=27)).strftime('%Y-%m-%d'),
                )
                articles = response.get('articles', [])
                for art in articles:
                    newsapi_headlines.append({
                        'date':     art['publishedAt'][:10],      # YYYY-MM-DD only
                        'headline': art['title'] or '',
                        'source':   art.get('source', {}).get('name', 'unknown'),
                    })
                print(f"  '{query[:40]}' → {len(articles)} headlines")
                time.sleep(0.5)  # be polite to the API
                break
            except Exception as e:
                print(f'  Attempt {attempt}/3 failed: {e}')
                if attempt < 3:
                    time.sleep(2 ** attempt)

    newsapi_df = pd.DataFrame(newsapi_headlines)
    print(f'\nTotal raw headlines from NewsAPI: {len(newsapi_df)}')
    if len(newsapi_df) > 0:
        newsapi_df['date'] = pd.to_datetime(newsapi_df['date'])
        newsapi_df = newsapi_df.dropna(subset=['headline'])
        newsapi_df = newsapi_df[newsapi_df['headline'].str.strip() != '']
        print(f'Headlines after cleaning: {len(newsapi_df)}')
        print(f'Date range: {newsapi_df["date"].min().date()} to {newsapi_df["date"].max().date()}')
        newsapi_df.head(3)

=== Fetching NewsAPI Recent Headlines ===
  'VIX volatility index' → 99 headlines
  'Federal Reserve interest rates' → 97 headlines
  'stock market crash recession' → 34 headlines
  'inflation CPI unemployment' → 47 headlines
  'S&P 500 market selloff' → 99 headlines

Total raw headlines from NewsAPI: 376
Headlines after cleaning: 376
Date range: 2026-02-28 to 2026-03-26


In [16]:
#Kaggle Historical Headlines (Full History)
KAGGLE_FILE = '../data/raw/analyst_ratings_processed.csv'

kaggle_df = None

if os.path.exists(KAGGLE_FILE):
    print(f'Loading Kaggle dataset: {KAGGLE_FILE}')
    kaggle_raw = pd.read_csv(KAGGLE_FILE)
    print(f'Raw shape: {kaggle_raw.shape}')
    print(f'Columns: {list(kaggle_raw.columns)}')

    headline_col = next(
        (c for c in ['title', 'headline', 'text', 'Title', 'Headline'] if c in kaggle_raw.columns), None
    )
    date_col = next(
        (c for c in ['date', 'Date', 'published', 'publishedAt', 'pub_date'] if c in kaggle_raw.columns), None
    )

    print(f'Detected headline column: {headline_col}')
    print(f'Detected date column:     {date_col}')

    if headline_col and date_col:
        kaggle_df = kaggle_raw[[headline_col, date_col]].rename(
            columns={headline_col: 'headline', date_col: 'date'}
        )
        kaggle_df['source'] = 'kaggle'

        # Parse dates — utc=True handles mixed tz-aware and tz-naive values
        kaggle_df['date'] = pd.to_datetime(kaggle_df['date'], errors='coerce', utc=True)

        # Strip timezone so dtype becomes datetime64[ns] not datetime64[ns, UTC]
        kaggle_df['date'] = kaggle_df['date'].dt.tz_localize(None)

        # Drop rows where date or headline is null
        before = len(kaggle_df)
        kaggle_df = kaggle_df.dropna(subset=['date', 'headline'])
        print(f'Rows dropped (bad dates/headlines): {before - len(kaggle_df):,}')

        # Normalize to midnight
        kaggle_df['date'] = kaggle_df['date'].dt.normalize()

        # Filter to project date range
        kaggle_df = kaggle_df[kaggle_df['date'] >= '2000-01-01']

        print(f'Headlines after cleaning: {len(kaggle_df):,}')
        print(f'Date range: {kaggle_df["date"].min().date()} to {kaggle_df["date"].max().date()}')
        print(kaggle_df.head(5))
    else:
        print(f'Could not detect columns. Available: {list(kaggle_raw.columns)}')
else:
    print(f'File not found: {KAGGLE_FILE}')

Loading Kaggle dataset: ../data/raw/analyst_ratings_processed.csv
Raw shape: (1400469, 4)
Columns: ['Unnamed: 0', 'title', 'date', 'stock']
Detected headline column: title
Detected date column:     date
Rows dropped (bad dates/headlines): 2,578
Headlines after cleaning: 1,397,891
Date range: 2009-02-14 to 2020-06-11
                                            headline       date  source
0            Stocks That Hit 52-Week Highs On Friday 2020-06-05  kaggle
1         Stocks That Hit 52-Week Highs On Wednesday 2020-06-03  kaggle
2                      71 Biggest Movers From Friday 2020-05-26  kaggle
3       46 Stocks Moving In Friday's Mid-Day Session 2020-05-22  kaggle
4  B of A Securities Maintains Neutral on Agilent... 2020-05-22  kaggle


In [19]:
##  Merge and Deduplicate News Sources
frames_to_merge = []

# Add NewsAPI frame 
if len(newsapi_headlines) > 0 and 'newsapi_df' in dir() and len(newsapi_df) > 0:
    newsapi_df['source'] = 'newsapi'
    frames_to_merge.append(newsapi_df[['date', 'headline', 'source']])
    print(f'NewsAPI headlines to merge: {len(newsapi_df):,}')

# Add Kaggle frame
if kaggle_df is not None:
    frames_to_merge.append(kaggle_df[['date', 'headline', 'source']])
    print(f'Kaggle headlines to merge:  {len(kaggle_df):,}')

if len(frames_to_merge) == 0:
    print('No news data loaded. Creating empty placeholder for Phase 3.')
    news_df = pd.DataFrame(columns=['date', 'headline', 'source'])
else:
    news_df = pd.concat(frames_to_merge, ignore_index=True)
    news_df['date'] = pd.to_datetime(news_df['date'])

    
    news_df = news_df[news_df['headline'].notna()]
    news_df = news_df[news_df['headline'].str.strip().str.len() > 10]

    before_dedup = len(news_df)

    # Deduplication: same date + same first 80 characters = duplicate
    news_df['dedup_key'] = (
        news_df['date'].astype(str) + '|' +
        news_df['headline'].str[:80].str.lower().str.strip()
    )
    news_df = news_df.drop_duplicates(subset='dedup_key').drop(columns='dedup_key')
    after_dedup = len(news_df)

    print(f'\nBefore deduplication: {before_dedup:,}')
    print(f'After  deduplication: {after_dedup:,}')
    print(f'Duplicates removed:   {before_dedup - after_dedup:,}')

    # Sort by date
    news_df = news_df.sort_values('date').reset_index(drop=True)

    print(f'\nFinal news dataset: {len(news_df):,} headlines')
    if len(news_df) > 0:
        print(f'Date range: {news_df["date"].min().date()} to {news_df["date"].max().date()}')

news_df.head()

NewsAPI headlines to merge: 376
Kaggle headlines to merge:  1,397,891

Before deduplication: 1,398,197
After  deduplication: 870,608
Duplicates removed:   527,589

Final news dataset: 870,608 headlines
Date range: 2009-02-14 to 2026-03-26


,date,headline,source
0,2009-02-14,How Treasuries and ETFs Work,kaggle
1,2009-04-27,Update on the Luxury Sector: 2nd Quarter 2009,kaggle
2,2009-04-29,Going Against the Herd,kaggle
3,2009-05-22,Charles Sizemore Radio Interview Saturday Morning,kaggle
4,2009-05-27,"JVA perks to 39% gain, SMCG ready, MRM to cont...",kaggle


In [20]:
NEWS_PATH = '../data/raw/news_raw.csv'

news_df.to_csv(NEWS_PATH, index=False)
print(f'Saved: {NEWS_PATH}')
print(f'Rows: {len(news_df):,}')
print(f'Columns: {list(news_df.columns)}')

# Log a source breakdown
if len(news_df) > 0:
    source_counts = news_df['source'].value_counts()
    print('\nHeadlines by source:')
    for src, count in source_counts.items():
        print(f'  {src:<12}: {count:>8,}')

    # Coverage check: what % of trading days have at least one headline?
    trading_days = pd.bdate_range(start='2000-01-01', end=news_df['date'].max())
    days_with_news = news_df['date'].dt.normalize().nunique()
    coverage_pct = days_with_news / len(trading_days) * 100
    print(f'\nNews coverage: {days_with_news:,} days have headlines')
    print(f'Out of ~{len(trading_days):,} trading days = {coverage_pct:.1f}% coverage')
    print('Days without headlines will receive sentiment = 0.0 (neutral) in Phase 3')
else:
    print('\nNote: Empty news file saved. Phase 3 will use neutral sentiment (0.0) for all dates.')

Saved: ../data/raw/news_raw.csv
Rows: 870,608
Columns: ['date', 'headline', 'source']

Headlines by source:
  kaggle      :  870,250
  newsapi     :      358

News coverage: 3,982 days have headlines
Out of ~6,844 trading days = 58.2% coverage
Days without headlines will receive sentiment = 0.0 (neutral) in Phase 3


In [21]:
##  Ingestion Log
log_lines = [
    f'=== DATA INGESTION LOG ===',
    f'Run timestamp:   {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    f'',
    f'VIX',
    f'  Rows:          {len(vix_df)}',
    f'  Date range:    {vix_df.index[0].date()} to {vix_df.index[-1].date()}',
    f'  Source:        Yahoo Finance via yfinance',
    f'',
    f'FRED Macro',
    f'  Shape:         {macro_df.shape}',
    f'  Series:        {", ".join(macro_df.columns.tolist())}',
    f'  Date range:    {macro_df.index[0].date()} to {macro_df.index[-1].date()}',
    f'',
    f'News Headlines',
    f'  Total rows:    {len(news_df)}',
]

if len(news_df) > 0:
    for src, cnt in news_df['source'].value_counts().items():
        log_lines.append(f'  {src:<14} {cnt:,}')
    log_lines.append(f'  Date range:    {news_df["date"].min().date()} to {news_df["date"].max().date()}')

log_lines += [
    f'',
    f'Files saved:',
    f'  data/raw/vix_raw.csv',
    f'  data/raw/macro_raw.csv',
    f'  data/raw/news_raw.csv',
    f'  .env.example',
]

log_text = '\n'.join(log_lines)
with open('../logs/ingestion_log.txt', 'w') as f:
    f.write(log_text)

print(log_text)

=== DATA INGESTION LOG ===
Run timestamp:   2026-03-27 15:28:54

VIX
  Rows:          6598
  Date range:    2000-01-03 to 2026-03-27
  Source:        Yahoo Finance via yfinance

FRED Macro
  Shape:         (314, 6)
  Series:        fedfunds, cpi, unrate, gs10, indpro, m2sl
  Date range:    2000-01-01 to 2026-02-01

News Headlines
  Total rows:    870608
  kaggle         870,250
  newsapi        358
  Date range:    2009-02-14 to 2026-03-26

Files saved:
  data/raw/vix_raw.csv
  data/raw/macro_raw.csv
  data/raw/news_raw.csv
  .env.example


In [32]:
## Full Phase 2 Data Validation Report
print('=' * 60)
print('       PHASE 2 — FINAL DATA VALIDATION REPORT')
print('=' * 60)

checks_passed = 0
checks_total  = 0

def check(name, condition, detail=''):
    global checks_passed, checks_total
    checks_total += 1
    status = 'PASS' if condition else 'FAIL'
    if condition:
        checks_passed += 1
    flag = '' if condition else ' ***'
    print(f'  [{status}] {name}{flag}')
    if detail:
        print(f'         {detail}')

print('\n--- VIX ---')
vix_check = pd.read_csv('../data/raw/vix_raw.csv', index_col=0, parse_dates=True)
check('vix_raw.csv exists',          os.path.exists('../data/raw/vix_raw.csv'))
check('VIX has data',                len(vix_check) > 5000,  f'{len(vix_check)} rows')
check('VIX starts from 2000',        vix_check.index[0].year <= 2000)
check('VIX has no NaN',              vix_check.isnull().sum().sum() == 0)

print('\n--- FRED Macro ---')
macro_check = pd.read_csv('../data/raw/macro_raw.csv', index_col=0, parse_dates=True)
check('macro_raw.csv exists',        os.path.exists('../data/raw/macro_raw.csv'))
check('All 6 FRED series present',   len(macro_check.columns) >= 6,  str(list(macro_check.columns)))
check('Macro starts from 2000',      macro_check.index[0].year <= 2000)
check('Macro NaN below 5%',          macro_check.isnull().mean().max() < 0.05)

print('\n--- News ---')
news_check = pd.read_csv('../data/raw/news_raw.csv')
check('news_raw.csv exists',         os.path.exists('../data/raw/news_raw.csv'))
check('News has headline column',    'headline' in news_check.columns)
check('News has date column',        'date' in news_check.columns)
check('No empty headlines',          news_check['headline'].isna().sum() == 0 if len(news_check) > 0 else True)

print('\n--- Security ---')
check('.env.example created',        os.path.exists('../.env.example'))
if os.path.exists('../.gitignore'):
    with open('../.gitignore') as f:
        gi = f.read()
    check('.env in .gitignore',      '.env' in gi)
else:
    check('.gitignore exists',       False, 'Create .gitignore before committing')

print(f'\n{"=" * 60}')
print(f'Result: {checks_passed}/{checks_total} checks passed')
if checks_passed == checks_total:
    print('All checks passed. Ready for Phase 3.')
else:
    print('Some checks failed (marked ***). Review above before continuing.')
print('=' * 60)

       PHASE 2 — FINAL DATA VALIDATION REPORT

--- VIX ---
  [PASS] vix_raw.csv exists
  [PASS] VIX has data
         6598 rows
  [PASS] VIX starts from 2000
  [PASS] VIX has no NaN

--- FRED Macro ---
  [PASS] macro_raw.csv exists
  [PASS] All 6 FRED series present
         ['fedfunds', 'cpi', 'unrate', 'gs10', 'indpro', 'm2sl']
  [PASS] Macro starts from 2000
  [PASS] Macro NaN below 5%

--- News ---
  [PASS] news_raw.csv exists
  [PASS] News has headline column
  [PASS] News has date column
  [PASS] No empty headlines

--- Security ---
  [PASS] .env.example created
  [PASS] .env in .gitignore

Result: 14/14 checks passed
All checks passed. Ready for Phase 3.


In [27]:
# Missing data audit
print('=== Missing Data Audit (Raw Files) ===')
print('\n--- FRED Macro (expected: monthly gaps in daily view) ---')
print(macro_df.isnull().sum())
print(f'\nNote: All NaNs here are expected; FRED is monthly frequency.')
print('Phase 3 will forward-fill to daily using .resample().ffill()')

print('\n--- VIX ---')
print(vix_df.isnull().sum())
print('Expected: 0 NaNs (dropna already applied in download)')

print('\n--- News ---')
if len(news_df) > 0:
    trading_days = len(pd.bdate_range('2000-01-01', news_df['date'].max()))
    days_with_news = news_df['date'].dt.normalize().nunique()
    gap_pct = (1 - days_with_news / trading_days) * 100
    print(f'Trading days with NO headlines: {gap_pct:.1f}%')
    print(f'Pre-2009 days will all be neutral (0.0);  Kaggle coverage starts 2009')
    print('Phase 3 fix: fill missing sentiment with 0.0 after FinBERT inference')

=== Missing Data Audit (Raw Files) ===

--- FRED Macro (expected: monthly gaps in daily view) ---
fedfunds    0
cpi         1
unrate      1
gs10        0
indpro      0
m2sl        0
dtype: int64

Note: All NaNs here are expected; FRED is monthly frequency.
Phase 3 will forward-fill to daily using .resample().ffill()

--- VIX ---
VIX    0
dtype: int64
Expected: 0 NaNs (dropna already applied in download)

--- News ---
Trading days with NO headlines: 41.8%
Pre-2009 days will all be neutral (0.0);  Kaggle coverage starts 2009
Phase 3 fix: fill missing sentiment with 0.0 after FinBERT inference


## Phase 2 Summary

### What Was Collected

| File | Contents | Used in |
|------|----------|---------|
| `data/raw/vix_raw.csv` | Daily VIX close, 2000–present | Phase 3 cleaning |
| `data/raw/macro_raw.csv` | 6 FRED monthly macro series | Phase 3 cleaning |
| `data/raw/news_raw.csv` | Financial headlines (Kaggle + NewsAPI) | Phase 3 FinBERT |
| `.env.example` | API key template | All phases |
| `logs/ingestion_log.txt` | Timestamped record of this run | Reference |

### What Phase 3 Does With These Files

- `vix_raw.csv` → compute lag features (1, 5, 21 days) and rolling stats
- `macro_raw.csv` → forward-fill monthly values to daily frequency, normalize
- `news_raw.csv` → run FinBERT to produce `sentiment_scores.csv`
- All three → merge into `data/processed/master_df.csv`
